In [20]:
# 1) Imports & Paths
import pandas as pd
from pathlib import Path

ROOT      = Path().resolve().parents[0]
RAW_DIR   = ROOT / "data" / "raw"
CLEAN_DIR = ROOT / "data" / "clean"

In [21]:
# 2) Load raw nursing‐home data
raw_path = RAW_DIR / "nh_data.csv"
df = pd.read_csv(raw_path, dtype=str)

In [22]:
# 3) Parse processing_date and drop unparseable rows
df["processing_date"] = pd.to_datetime(df["processing_date"], errors="coerce")
df = df.dropna(subset=["processing_date"])

In [23]:
# 4) Restrict to analysis window (Jan 2018 – Jul 2025)
df = df[
    (df["processing_date"] >= "2018-01-01") &
    (df["processing_date"] <= "2025-07-31")
]

In [24]:
# 5) Cast key numeric columns
numeric_cols = [
    "number_of_certified_beds"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [25]:
# 6) Drop columns that are entirely null (e.g., empty rating fields)
df = df.dropna(axis=1, how="all")

In [26]:
# 7) Drop exact duplicates by (CMS ID, processing_date)
df = df.drop_duplicates(subset=["cms_certification_number", "processing_date"])

In [27]:
# 8) Create a month‐stamp for merging
df["report_month"] = df["processing_date"].dt.to_period("M").dt.to_timestamp()

In [28]:
# 9) Save cleaned dataset
output_path = CLEAN_DIR / "nh_data_clean.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned nursing‐home rows: {len(df)}")
df.head(n=100)

Cleaned nursing‐home rows: 144


,cms_certification_number,provider_name,number_of_certified_beds,processing_date,report_month
0,366480,TAYLOR SPRINGS HEALTH CAMPUS,50,2021-03-01,2021-03-01
1,366480,TAYLOR SPRINGS HEALTH CAMPUS,50,2021-04-01,2021-04-01
2,366480,TAYLOR SPRINGS HEALTH CAMPUS,50,2021-05-01,2021-05-01
3,366480,TAYLOR SPRINGS HEALTH CAMPUS,50,2021-06-01,2021-06-01
4,366480,TAYLOR SPRINGS HEALTH CAMPUS,50,2021-07-01,2021-07-01
...,...,...,...,...,...
107,NaN,SAGE PARK FACILITY,50,2021-07-01,2021-07-01
108,NaN,SAGE PARK FACILITY,50,2021-08-01,2021-08-01
109,NaN,SAGE PARK FACILITY,50,2021-09-01,2021-09-01
110,NaN,SAGE PARK FACILITY,50,2021-10-01,2021-10-01
